# pyDNEMD Workshop

This workshop will introduce you how to set up and analyse **dynamical non-equilibrium simulations** using Gromacs and `pyDNEMD`. It will also provide a quick overview of dynamical-nonequilibrium simulations and how such an approach can be used to study allostery in proteins.

`pyDNEMD` can be downloaded from `https://github.com/lorenzo-tulli/pyDNEMD.git` following the instructions in the README file.


## 0. Prerequisites

To open the container, run the following commands in a power shell in your local computer with **docker desktop previously installed**.

```
docker pull ghcr.io/lorenzo-tulli/pydnemd-workshop:latest
docker run -p 8888:8888 ghcr.io/lorenzo-tulli/pydnemd-workshop:latest
```

When the container is running, open:

```
http://localhost:8888/
```


### What is D-NEMD?

D-NEMD reveals how a protein responds to a sudden perturbation (e.g. removing a ligand).  
The experiment runs three sets of simulations:

| Leg | Description |
|-----|-------------|
| **EQ** | Long equilibrium production run — provides starting snapshots for non-equilibrium and null perturbation|
| **NE** | Non-Equilibrium: perturbed topology, velocities kept from EQ |
| **NP** | Null-Perturbation: original topology, velocities reassigned |

The displacement vectors calculated as **(NE − NP)** removes thermal noise highlighting the response of the system to the applied perturbation.

### Workflow

```
1. Run equilibrium simulations          dnemd-run-equilibrium
2. Analyse equilibrium production       dnemd-analyse-equilibrium
3. Create NE/NP input files             dnemd-create-ne-np
4. Run NE and NP simulations            dnemd-run-ne / dnemd-run-np
5. Extract Cα frames                    dnemd-extract
6. Analyse D-NEMD displacement          dnemd-analyse-dnemd
```

Before you start, make sure the following input files are ready:

| File | Description |
|------|-------------|
| `solv_ions.gro` | Solvated, ionised starting structure |
| `topol.top` | GROMACS topology |
| `index.ndx` | Index file with standard groups |
| `templates/` | Directory containing MDP files (`em.mdp`, `step1.mdp` … `step4.mdp`, `prod.mdp`) |

Edit `config.yaml` to point to your files:

In [ ]:
# View the config file
!cat config.yaml

The key fields to update in `config.yaml`:

```yaml
input_gro:  /path/to/solv_ions.gro
topology:   /path/to/topol.top
index_ndx:  /path/to/index.ndx
mdp_dir:    /path/to/templates
output_dir: /path/to/output
system_name: MY_SYSTEM
gmx: gmx_mpi          # or 'gmx' for a local installation
n_runs: N              # number of independent EQ replicates
```

Set the config path here once and it will be reused throughout the notebook:

In [ ]:
CONFIG = "config.yaml"

---
## 1. Run equilibrium simulations

`dnemd-equilibrium` sets up and runs the full GROMACS equilibration pipeline for each replicate:

```
energy minimisation → NVT (heavy atoms) → NVT (Cα) → NPT (Cα) → NPT (backbone) → production
```

Output is written to `output/EQ_{run}/`.

> **On HPC**: submit each run as a separate SLURM job and use `--run N` to select the replicate.

For the purpose of this tutorial the simulations have already been runned. The data can be found in /examples/output

In [ ]:
# Run all replicates sequentially (n_runs from config)
!dnemd-equilibrium --config $CONFIG

'dnemd-run-equilibrium' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
# Or run a single replicate (e.g. run 1)
!dnemd-run-equilibrium --config $CONFIG --run 1

In [ ]:
# Prepare input files only — without launching GROMACS (useful for inspection)
!dnemd-run-equilibrium --config $CONFIG --setup-only

---
## 2. Analyse equilibrium production

`dnemd-analyse-equilibrium` computes Cα **RMSD** and **RMSF** for each replicate using the energy-minimised structure as reference, then writes plots and a summary CSV.

Outputs in `output/results/equilibrium/`:
- `rmsd_run{run}.xvg` — raw GROMACS RMSD data
- `rmsf_run{run}.xvg` — raw GROMACS RMSF data
- `rmsd_all_runs.png` — overlaid RMSD plot
- `rmsf_all_runs.png` — overlaid RMSF plot
- `summary.csv` — per-run mean/max RMSD and RMSF

> Use these plots to confirm convergence and choose which frames to use as NE/NP starting points.

In [ ]:
# Analyse all runs
!dnemd-analyse-equilibrium --config $CONFIG

In [ ]:
# Or analyse a single run
!dnemd-analyse-equilibrium --config $CONFIG --run 1

In [ ]:
# Display the RMSD and RMSF plots inline
from IPython.display import Image, display
import yaml

with open(CONFIG) as f:
    cfg = yaml.safe_load(f)

results_eq = f"{cfg['output_dir']}/results/equilibrium"

display(Image(f"{results_eq}/rmsd_all_runs.png"))
display(Image(f"{results_eq}/rmsf_all_runs.png"))

In [ ]:
# Print the summary table
import pandas as pd
pd.read_csv(f"{results_eq}/summary.csv")

---
## 3. Create NE/NP input files

Before running this step, check the files in `output/perturbed_topology/` exist:

| File | How to create |
|------|---------------|
| `extraction_index.ndx` | Copy of `index.ndx` with an extra group `Protein_Water_and_ions` (protein + solvent, ligand excluded). Create it with `gmx_mpi make_ndx`. |
| `topolperturb.top` | Copy of `topol.top` with the ligand removed from `[ molecules ]`. |

**Creating `extraction_index.ndx`:**
```bash
gmx_mpi make_ndx -f output/EQ_1/em/em.gro \
                 -n inputs/index.ndx \
                 -o output/perturbed_topology/extraction_index.ndx
# In the make_ndx prompt:
#   1 | 20          <- combine Protein group with Water_and_ions group
#   name N Protein_Water_and_ions
#   q
```

`dnemd-create-ne-np` will then:
1. Auto-generate `perturbed_index.ndx` from the perturbed topology
2. Extract EQ frames at the specified interval
3. Create `.tpr` files for all NE and NP simulations

Output layout:
```
output/
  NE_{run}/{time}ns/   MD_NE.tpr, Prod_RunNE.mdp
  NP_{run}/{time}ns/   MD_NP.tpr, Prod_RunNP.mdp
```

In [ ]:
# Create all NE/NP input files (frame range from config)
!dnemd-create-ne-np --config $CONFIG --skip-topology-test

In [ ]:
# Override the frame range from the command line if needed
# --start / --end / --frequency are in picoseconds
!dnemd-create-ne-np --config $CONFIG --start 50000 --end 500000 --frequency 5000

In [ ]:
# Process only a single EQ run
!dnemd-create-ne-np --config $CONFIG --run 1

---
## 4. Run NE and NP simulations

`dnemd-run-ne` and `dnemd-run-np` each run a single 1 ns GROMACS simulation for one `(run, time)` combination.

> **On HPC**: submit as SLURM array jobs — one task per `(run, time_ns)` pair.  
> The `--task-id` flag maps the array index to the correct run and time point automatically.

The number of tasks needed is `n_runs × n_time_points`.

For the purpose of this tutorial the simulations have already been runned. The data can be found in /examples/output

In [ ]:
# Run a single NE simulation (run 1, starting from the 100 ns EQ frame)
!dnemd-run-ne --config $CONFIG --run 1 --time-ns 100

In [ ]:
# Run the corresponding NP control simulation
!dnemd-run-np --config $CONFIG --run 1 --time-ns 100

In [ ]:
# SLURM array mode — task ID is mapped automatically to (run, time_ns)
# !dnemd-run-ne --config $CONFIG --task-id $SLURM_ARRAY_TASK_ID
# !dnemd-run-np --config $CONFIG --task-id $SLURM_ARRAY_TASK_ID

---
## 5. Extract Cα frames

`dnemd-extract` extracts Cα frames from **all three legs** (EQ, NE, NP) at the requested ps time points within each ns window.

For each frame it runs `trjconv` with `-fit rot+trans` (fitting group = Protein, output group = C-alpha).  
For NE/NP, it also centers and fixes PBC before dumping frames.

Output naming: `Run{run}{leg}{ns}ns{ps}ps.gro`  
e.g. `output/NE_1/100ns/Run1NE100ns1000ps.gro`

> **On HPC**: submit as a SLURM array with one task per ns window.

For the purpose of this tutorial, the frames have already been extracted.

In [ ]:
# Extract all legs for all ns windows sequentially
!dnemd-extract --config $CONFIG --all

In [ ]:
# Extract a single ns window (useful for testing)
!dnemd-extract --config $CONFIG --ns 100

In [ ]:
# Extract only one leg
!dnemd-extract --config $CONFIG --all --leg ne
!dnemd-extract --config $CONFIG --all --leg np
!dnemd-extract --config $CONFIG --all --leg eq

In [ ]:
# Override ps time points to extract within each ns window
!dnemd-extract --config $CONFIG --all --ps-timepoints 0 100 500 1000

---
## 6. Analyse D-NEMD displacement vectors

Before running this step, check the reference structure at `output/analysis_reference.pdb` exist.  
This can be the frame closest to the average structure across all production runs.

**Finding the closest-to-average frame with MDAnalysis:**
```python
import MDAnalysis as mda
import numpy as np

u   = mda.Universe('prod.tpr', 'prod.xtc')
ca  = u.select_atoms('name CA')
pos = np.array([ca.positions.copy() for ts in u.trajectory])

mean_pos   = pos.mean(axis=0)
rmsd       = np.sqrt(((pos - mean_pos)**2).sum(axis=(1,2)) / len(ca))
best_frame = int(rmsd.argmin())

u.trajectory[best_frame]
ca.write('output/analysis_reference.pdb')
```

`dnemd-analyse-dnemd` then:
1. Collects displacement vectors **(NE − NP)** across all runs and time points
2. Filters by significance (standard error threshold)
3. Writes per-Cα statistics and a B-factor PDB coloured by displacement magnitude

Outputs in `output/results/dnemd/`:
- `stat_{system}_{tp}ps_{N}SE.txt` — per-Cα statistics table
- `vec_norm_{system}_{tp}ps_{N}SE.pdb` — Cα PDB with B-factors = |displacement|
- `summary.csv` — aggregated stats across all time points

In [ ]:
# Analyse all time points defined in config
!dnemd-analyse-dnemd --config $CONFIG

In [ ]:
# Analyse a single time point (e.g. 1000 ps after perturbation)
!dnemd-analyse-dnemd --config $CONFIG --time-point 1000

In [ ]:
# Use a custom reference PDB
!dnemd-analyse-dnemd --config $CONFIG --ref-pdb /path/to/my_reference.pdb

In [ ]:
# Print the summary of results
import pandas as pd
import yaml

with open(CONFIG) as f:
    cfg = yaml.safe_load(f)

results_dnemd = f"{cfg['output_dir']}/results/dnemd"
pd.read_csv(f"{results_dnemd}/summary.csv")

---
## 7. Visualise results

The B-factor PDB files (`vec_norm_*.pdb`) can be opened directly in PyMOL or VMD and coloured by B-factor to visualise which Cα atoms respond significantly to the perturbation.

**PyMOL:**
```
load output/results/dnemd/vec_norm_SYSTEM_1000ps_1SE.pdb
spectrum b, blue_white_red, minimum=0, maximum=2
```

**VMD:**
```
mol new output/results/dnemd/vec_norm_SYSTEM_1000ps_1SE.pdb
mol modcolor 0 0 Beta
```